# VoiceCraft AI — Applio CPU setup
This notebook prepares a temporary Google Colab CPU runtime. **On standard Colab, CPU is the default** when no GPU/TPU accelerator is selected. If the mobile menu does not show Runtime settings, you can leave the default runtime as-is: tap Connect, then run the first code cell. Do not select GPU or TPU.

**Conversion requirements:** an authorized Applio inference `.pth` model (and optional matching `.index`). This notebook does not download unlicensed voice models.

**Important:** Colab is temporary testing only, not an always-on production server. Run cells in order and inspect errors before continuing.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg aria2
!git clone --depth 1 --recurse-submodules https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio
!bash run-install.sh

## Verify CPU inference CLI
If this cell fails, stop and inspect the error before continuing.

In [ ]:
%cd /content/Applio
!python core.py infer --help

## Next: add an authorized inference voice model
Applio is installed, but conversion cannot run until you provide an inference `.pth` model that you are authorized to use. Upload the matching `.index` too if the model publisher supplied one. Do not upload training checkpoints such as `G_*.pth`.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()  # Choose an authorized inference .pth and optional matching .index
model_dir = Path('/content/Applio/logs/VoiceCraftModels')
model_dir.mkdir(parents=True, exist_ok=True)
saved = []
for name in uploaded:
    suffix = Path(name).suffix.lower()
    if suffix not in {'.pth', '.index'}:
        print(f'Skipped unsupported file: {name}')
        continue
    destination = model_dir / Path(name).name
    shutil.move(name, destination)
    saved.append(destination)
for item in saved:
    print(f'Saved: {item} ({item.stat().st_size:,} bytes)')
pth_files = [p for p in saved if p.suffix.lower() == '.pth']
if not pth_files:
    print('No .pth inference model uploaded yet. Upload one before conversion testing.')
else:
    print('Inference model file received. Confirm it is an inference export, not a training checkpoint.')


## Important
This is only the Applio installation and CLI verification stage. VoiceCraft's API worker needs a reachable HTTPS runtime, a strong server-side token, and a permitted `.pth` inference model. Colab sessions are temporary; do not treat this notebook as production hosting.